# I06 - Object Detection and Segmentation

**Level:** Intermediate

---

## Learning Objectives

By the end of this lesson, you will:
1. Understand object detection vs classification
2. Learn YOLO architecture and concepts
3. Explore R-CNN family (Fast R-CNN, Faster R-CNN)
4. Implement semantic segmentation with U-Net
5. Understand instance segmentation (Mask R-CNN)

---

## Prerequisites

- Completed I04-I05 (CNN Architectures, Transfer Learning)
- Strong understanding of CNNs
- Familiarity with bounding boxes and IoU

---

In [ ]:
# Import required libraries
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print(f"TensorFlow version: {tf.__version__}")

## 1. Computer Vision Tasks Hierarchy

### Task Comparison

1. **Image Classification**
   - Input: Image
   - Output: Class label
   - Example: "This is a cat"

2. **Object Localization**
   - Input: Image
   - Output: Class + Bounding box
   - Example: "Cat at (x, y, w, h)"

3. **Object Detection**
   - Input: Image
   - Output: Multiple classes + bounding boxes
   - Example: "Cat at (x1, y1, w1, h1), Dog at (x2, y2, w2, h2)"

4. **Semantic Segmentation**
   - Input: Image
   - Output: Pixel-wise class labels
   - Example: Every pixel labeled as cat, dog, background, etc.

5. **Instance Segmentation**
   - Input: Image
   - Output: Pixel-wise labels + instance IDs
   - Example: Cat1, Cat2, Dog1 (separate instances)

## 2. Object Detection Fundamentals

### Key Concepts

**Bounding Box:**
- Represented as (x, y, width, height) or (x1, y1, x2, y2)
- x, y: top-left corner coordinates
- width, height: box dimensions

**Intersection over Union (IoU):**
$$IoU = \frac{Area\ of\ Overlap}{Area\ of\ Union}$$

**Non-Maximum Suppression (NMS):**
- Remove duplicate detections
- Keep box with highest confidence
- Suppress overlapping boxes (IoU > threshold)

**Anchor Boxes:**
- Pre-defined box shapes
- Different aspect ratios and scales
- Model predicts offsets from anchors

In [ ]:
def calculate_iou(box1, box2):
    """Calculate Intersection over Union (IoU) between two boxes
    
    Args:
        box1, box2: [x1, y1, x2, y2] format
    
    Returns:
        IoU score (0 to 1)
    """
    # Calculate intersection
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    
    # Calculate union
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = box1_area + box2_area - intersection
    
    return intersection / union if union > 0 else 0

# Example
box_a = [10, 10, 50, 50]
box_b = [30, 30, 70, 70]
iou = calculate_iou(box_a, box_b)
print(f"IoU between boxes: {iou:.3f}")

# Visualize
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
rect1 = patches.Rectangle((box_a[0], box_a[1]), box_a[2]-box_a[0], box_a[3]-box_a[1],
                          linewidth=2, edgecolor='blue', facecolor='none', label='Box A')
rect2 = patches.Rectangle((box_b[0], box_b[1]), box_b[2]-box_b[0], box_b[3]-box_b[1],
                          linewidth=2, edgecolor='red', facecolor='none', label='Box B')
ax.add_patch(rect1)
ax.add_patch(rect2)
ax.set_xlim(0, 80)
ax.set_ylim(0, 80)
ax.set_aspect('equal')
ax.legend()
ax.set_title(f'IoU = {iou:.3f}', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.show()

## 3. YOLO: You Only Look Once

### YOLO Architecture

**Key Innovation:** Single-stage detector
- Divide image into grid (e.g., 7×7)
- Each cell predicts bounding boxes and class probabilities
- Very fast (real-time detection)

**YOLO Output:**
- For each grid cell:
  - B bounding boxes (x, y, w, h, confidence)
  - C class probabilities
- Total output: S × S × (B × 5 + C)

**Advantages:**
- Fast (45+ FPS)
- Sees entire image (good context)
- Fewer false positives

**Disadvantages:**
- Struggles with small objects
- Lower accuracy than two-stage detectors

In [ ]:
def create_yolo_style_detector(input_shape=(224, 224, 3), grid_size=7, 
                               num_boxes=2, num_classes=10):
    """Simplified YOLO-style detector
    
    Args:
        input_shape: Input image shape
        grid_size: Grid size (S x S)
        num_boxes: Number of boxes per grid cell (B)
        num_classes: Number of classes (C)
    
    Returns:
        Model that outputs (grid_size, grid_size, num_boxes * 5 + num_classes)
    """
    inputs = keras.Input(shape=input_shape)
    
    # Backbone (feature extractor)
    x = layers.Conv2D(64, 7, strides=2, padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D(2)(x)
    
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D(2)(x)
    
    x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D(2)(x)
    
    x = layers.Conv2D(512, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(512, 3, padding='same', activation='relu')(x)
    
    # Detection head
    x = layers.Conv2D(1024, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(1024, 3, padding='same', activation='relu')(x)
    
    # Output: (grid_size, grid_size, num_boxes * 5 + num_classes)
    output_channels = num_boxes * 5 + num_classes
    x = layers.Conv2D(output_channels, 1, padding='same')(x)
    
    # Reshape to grid
    outputs = layers.Reshape((grid_size, grid_size, output_channels))(x)
    
    model = Model(inputs, outputs, name='YOLO_Style')
    return model

yolo_model = create_yolo_style_detector()
print("YOLO-style detector created")
print(f"Output shape: {yolo_model.output_shape}")
print(f"\nInterpretation: (batch, {yolo_model.output_shape[1]}x{yolo_model.output_shape[2]} grid, "
      f"{yolo_model.output_shape[3]} channels)")
print(f"Channels = 2 boxes × 5 (x,y,w,h,conf) + 10 classes = 20")

## 4. R-CNN Family: Two-Stage Detectors

### Evolution of R-CNN

**1. R-CNN (2014)**
- Selective search for region proposals (~2000)
- CNN for each region
- Very slow (47s per image)

**2. Fast R-CNN (2015)**
- Single CNN for entire image
- RoI pooling for regions
- Faster (2s per image)

**3. Faster R-CNN (2015)**
- Region Proposal Network (RPN)
- End-to-end trainable
- Real-time capable (0.2s per image)

### Faster R-CNN Architecture

1. **Backbone CNN**: Extract features
2. **Region Proposal Network (RPN)**: Propose regions
3. **RoI Pooling**: Fixed-size features for each region
4. **Detection Head**: Classify and refine boxes

In [ ]:
def region_proposal_network(feature_map, num_anchors=9):
    """Simplified Region Proposal Network
    
    Args:
        feature_map: Feature map from backbone
        num_anchors: Number of anchor boxes per location
    
    Returns:
        objectness_scores: (H, W, num_anchors, 2) - object/background
        bbox_deltas: (H, W, num_anchors, 4) - box refinements
    """
    # Shared convolution
    x = layers.Conv2D(512, 3, padding='same', activation='relu')(feature_map)
    
    # Objectness classification (object vs background)
    objectness = layers.Conv2D(num_anchors * 2, 1, padding='same')(x)
    objectness = layers.Reshape((-1, 2))(objectness)
    objectness = layers.Activation('softmax')(objectness)
    
    # Bounding box regression
    bbox_deltas = layers.Conv2D(num_anchors * 4, 1, padding='same')(x)
    bbox_deltas = layers.Reshape((-1, 4))(bbox_deltas)
    
    return objectness, bbox_deltas

print("Region Proposal Network (RPN) function defined")
print("\nRPN predicts:")
print("1. Objectness scores: Is there an object?")
print("2. Bounding box deltas: How to adjust anchor boxes?")

## 5. Semantic Segmentation with U-Net

### U-Net Architecture

**Structure:**
- **Encoder (Contracting path)**: Downsample, extract features
- **Decoder (Expanding path)**: Upsample, recover spatial resolution
- **Skip connections**: Combine low and high-level features

**Key Features:**
- Symmetric encoder-decoder
- Skip connections preserve spatial information
- Works well with limited data
- Originally designed for medical imaging

**Applications:**
- Medical image segmentation
- Satellite image analysis
- Autonomous driving
- Background removal

In [ ]:
def conv_block(x, filters, kernel_size=3):
    """Convolution block: Conv -> BatchNorm -> ReLU -> Conv -> BatchNorm -> ReLU"""
    x = layers.Conv2D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    x = layers.Conv2D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    return x

def create_unet(input_shape=(256, 256, 3), num_classes=3):
    """Create U-Net model for semantic segmentation
    
    Args:
        input_shape: Input image shape
        num_classes: Number of segmentation classes
    
    Returns:
        U-Net model
    """
    inputs = keras.Input(shape=input_shape)
    
    # Encoder (Contracting path)
    c1 = conv_block(inputs, 64)
    p1 = layers.MaxPooling2D(2)(c1)
    
    c2 = conv_block(p1, 128)
    p2 = layers.MaxPooling2D(2)(c2)
    
    c3 = conv_block(p2, 256)
    p3 = layers.MaxPooling2D(2)(c3)
    
    c4 = conv_block(p3, 512)
    p4 = layers.MaxPooling2D(2)(c4)
    
    # Bottleneck
    c5 = conv_block(p4, 1024)
    
    # Decoder (Expanding path)
    u6 = layers.Conv2DTranspose(512, 2, strides=2, padding='same')(c5)
    u6 = layers.Concatenate()([u6, c4])  # Skip connection
    c6 = conv_block(u6, 512)
    
    u7 = layers.Conv2DTranspose(256, 2, strides=2, padding='same')(c6)
    u7 = layers.Concatenate()([u7, c3])  # Skip connection
    c7 = conv_block(u7, 256)
    
    u8 = layers.Conv2DTranspose(128, 2, strides=2, padding='same')(c7)
    u8 = layers.Concatenate()([u8, c2])  # Skip connection
    c8 = conv_block(u8, 128)
    
    u9 = layers.Conv2DTranspose(64, 2, strides=2, padding='same')(c8)
    u9 = layers.Concatenate()([u9, c1])  # Skip connection
    c9 = conv_block(u9, 64)
    
    # Output layer
    outputs = layers.Conv2D(num_classes, 1, activation='softmax')(c9)
    
    model = Model(inputs, outputs, name='U-Net')
    return model

unet_model = create_unet()
print("U-Net model created")
print(f"\nInput shape: {unet_model.input_shape}")
print(f"Output shape: {unet_model.output_shape}")
print(f"\nOutput: Pixel-wise class probabilities (same spatial size as input)")

In [ ]:
# Visualize U-Net architecture
print("\nU-Net Architecture Summary:")
print("=" * 70)
print("Encoder (Downsampling):")
print("  256x256 -> 128x128 -> 64x64 -> 32x32 -> 16x16")
print("\nBottleneck:")
print("  16x16 (deepest features)")
print("\nDecoder (Upsampling with skip connections):")
print("  16x16 -> 32x32 -> 64x64 -> 128x128 -> 256x256")
print("\nSkip connections preserve spatial details from encoder")
print("=" * 70)

## 6. Instance Segmentation: Mask R-CNN

### Mask R-CNN Architecture

**Extension of Faster R-CNN:**
- Adds mask prediction branch
- Predicts: class, bounding box, AND pixel mask
- Separate instances of same class

**Components:**
1. **Backbone**: Feature extraction (ResNet + FPN)
2. **RPN**: Region proposals
3. **RoI Align**: Better than RoI pooling (preserves spatial alignment)
4. **Detection Head**: Class + box
5. **Mask Head**: Pixel-wise segmentation

**Applications:**
- Autonomous driving
- Medical imaging
- Video analysis
- Robotics

## 7. Comparison of Detection Methods

### Performance Comparison

| Method | Type | Speed (FPS) | Accuracy | Use Case |
|--------|------|-------------|----------|----------|
| **YOLO** | One-stage | 45-155 | Good | Real-time applications |
| **SSD** | One-stage | 59 | Good | Mobile, embedded |
| **Faster R-CNN** | Two-stage | 7 | Excellent | High accuracy needed |
| **Mask R-CNN** | Two-stage | 5 | Excellent | Instance segmentation |
| **EfficientDet** | One-stage | 30 | Excellent | Best efficiency |

### Trade-offs

**One-stage detectors (YOLO, SSD):**
- ✅ Fast
- ✅ Simple architecture
- ❌ Lower accuracy
- ❌ Struggles with small objects

**Two-stage detectors (R-CNN family):**
- ✅ High accuracy
- ✅ Better with small objects
- ❌ Slower
- ❌ More complex

## Summary

In this lesson, you learned:

1. ✅ Object detection vs classification vs segmentation
2. ✅ YOLO architecture for real-time detection
3. ✅ R-CNN family evolution and Faster R-CNN
4. ✅ U-Net for semantic segmentation
5. ✅ Mask R-CNN for instance segmentation

**Key Takeaways:**
- Object detection = classification + localization
- YOLO is fast, R-CNN is accurate
- U-Net uses skip connections for segmentation
- Mask R-CNN combines detection and segmentation
- Choose method based on speed/accuracy requirements

**Next Steps:**
- Move to I07 - Advanced RNN Architectures (NLP track)
- Try pre-trained detection models (TensorFlow Object Detection API)
- Implement custom object detector for your use case

---

**References:**
- Redmon et al. (2016): "You Only Look Once: Unified, Real-Time Object Detection"
- Ren et al. (2015): "Faster R-CNN: Towards Real-Time Object Detection"
- Ronneberger et al. (2015): "U-Net: Convolutional Networks for Biomedical Image Segmentation"
- He et al. (2017): "Mask R-CNN"
- TensorFlow Object Detection API

---

## Learning Progress Tracker

Use this section to track your learning progress for this lesson.

### Completion Status
- [ ] Lesson completed
- [ ] All code cells executed successfully
- [ ] Understood all key concepts
- [ ] Completed practice exercises (if any)

### Dates
- **First Completed:** ____/____/____
- **Last Reviewed:** ____/____/____
- **Next Review:** ____/____/____ (Recommended: 1 week, 1 month, 3 months)

### Understanding Level
Rate your understanding (1-5): _____ / 5

- 1 = Need to review completely
- 2 = Understood basics, need more practice
- 3 = Good understanding, minor gaps
- 4 = Strong understanding, can explain to others
- 5 = Mastered, can apply in projects

### Notes & Reflections
```
Write your notes here:
- What concepts were challenging?
- What was interesting or surprising?
- How can you apply this in projects?
- Questions to explore further?




```

### Key Concepts to Remember (I06)
- [ ] YOLO architecture
- [ ] R-CNN family
- [ ] U-Net
- [ ] Mask R-CNN

---